<a href="https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yumna-Zafar/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Yumna-Zafar/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) in ("notebooks", "work"):
    os.chdir("..")
    if os.path.basename(os.getcwd()) == "work":
        os.chdir("..")

print("Working directory:", os.getcwd())

Working directory: /content/flyrank-ml-internship


In [2]:
%pip -q install duckdb huggingface_hub

In [3]:
import os, getpass

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

In [4]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':    f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':    f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':     f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_mar': f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    'fact_query_90d': f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}
print("Connected. Developing on month=2026-03 (mid-panel, not the sealed final month).")

Connected. Developing on month=2026-03 (mid-panel, not the sealed final month).


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Lane: Refresh / Content Opportunity Scoring (Lane 2).

Unit of analysis: one row = one content item, on one day (a "content-day"), from
fact_content_daily_performance. client_hash_id + content_hash_id + report_date
together identify one row.

Table(s) used: fact_content_daily_performance (daily signals) joined to dim_content
(content metadata) and dim_clients (per-client history coverage, checked before
trusting any window).

Time window: developing on the mid-panel month month=2026-03 (2026-03-01 to
2026-03-31), avoiding the sealed final month (June 2026) and the _sample table,
which IS the final month. Real feature building later will use a trailing window
(e.g. 90 days), not a single calendar month.

Label/proxy: whether a content item's impressions later declined, built as a real
future-window comparison (trailing period vs. following period) rather than the
starter CSV's current-window trend_direction bucket.

One thing deliberately excluded: any product-computed decision flag (health_score,
priority_score, action_type) -- not shipped in this dataset on purpose. Also
excluding raw query/URL/title fields -- only hashed IDs are touched, for joining.

In [5]:
row_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date
    FROM {TABLES['fact_daily_mar']}
    LIMIT 5
""").df()
row_check

,client_hash_id,content_hash_id,report_date
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (knowable before the decision moment):
- gsc_impressions, gsc_clicks, gsc_avg_position (past daily search signals)
- content_age_days, derived from dim_content.content_created_at (known at creation)
- rare_impressions_share, top_query_share (past query-mix signals, fact_content_query_90d)

Label / proxy:
- is_declining_proxy = a future-window decline signal, computed FROM impressions/
  position but only ever used as the target, never as an input feature.

Context (grouping/joining/splitting, never a model input):
- client_hash_id, content_hash_id, report_date, url_hash_id, keyword_hash_id

Excluded (with why):
- health_score, priority_score, action_type -- not in this dataset, and would be
  circular if they were (the product's own answer, not independent evidence).
- raw query/URL/title text -- scrambled before release; never available.
- the final month (June 2026) / _sample table -- reserved as a sealed test month.

In [6]:
print("Feature fields: gsc_impressions, gsc_clicks, gsc_avg_position, content_age_days, rare_share")
print("Label field:    is_declining_proxy (derived, future-window only)")
print("Context fields: client_hash_id, content_hash_id, report_date")
print("Excluded:       product decision flags, raw text fields, final sealed month")


Feature fields: gsc_impressions, gsc_clicks, gsc_avg_position, content_age_days, rare_share
Label field:    is_declining_proxy (derived, future-window only)
Context fields: client_hash_id, content_hash_id, report_date
Excluded:       product decision flags, raw text fields, final sealed month


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [7]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {TABLES['fact_daily_mar']}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Duplicate (client, content, date) rows found:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) rows found: 0


,client_hash_id,content_hash_id,report_date,c


In [8]:
span_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT content_hash_id) AS n_content_items,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {TABLES['fact_daily_mar']}
""").df()
span_check

,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


In [9]:
avail_check = con.sql(f"""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows
    FROM {TABLES['fact_daily_mar']}
""").df()
avail_check['pct_ga4_available'] = (avail_check['ga4_available_rows'] / avail_check['total_rows'] * 100).round(1)
avail_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_ga4_available
0,9841378,413966,4.2


In [11]:
feature_frame = con.sql(f"""
    SELECT
        f.content_hash_id,
        SUM(f.gsc_impressions) AS impressions_month,
        SUM(f.gsc_clicks) AS clicks_month,
        AVG(f.gsc_avg_position) AS avg_position_month,
        DATE_DIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
        ANY_VALUE(q.rare_impressions_share) AS rare_share
    FROM {TABLES['fact_daily_mar']} f
    LEFT JOIN {TABLES['dim_content']} c ON f.content_hash_id = c.content_hash_id
    LEFT JOIN {TABLES['fact_query_90d']} q ON f.content_hash_id = q.content_hash_id
    GROUP BY f.content_hash_id, c.content_created_date
    HAVING SUM(f.gsc_impressions) > 0
    LIMIT 2000
""").df()
print(feature_frame.shape)
feature_frame.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(2000, 6)


,content_hash_id,impressions_month,clicks_month,avg_position_month,content_age_days,rare_share
0,content_cec711b02f3bbde6,1806.0,12.0,4.428747,47,0.040972
1,content_614baf2af4330bd7,1544.0,2.0,4.685335,47,0.061244
2,content_755d951187fcd70a,26012.0,84.0,1.854929,47,0.032628
3,content_225dc9235023be5f,488.0,1.0,17.148172,47,0.292793
4,content_cdd114d71966c437,3776.0,0.0,10.400049,47,0.153882


In [12]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feature_frame['is_declining_proxy'] = (feature_frame['avg_position_month'] > 20).astype(int)

honest_features = ['impressions_month', 'clicks_month', 'content_age_days', 'rare_share']
X = feature_frame[honest_features].fillna(0)
y = feature_frame['is_declining_proxy']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
honest_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(X_te)[:, 1])
print(f"Honest ROC AUC (no leak): {honest_auc:.3f}")

# Deliberate leak: avg_position_month was used to DEFINE the label
leaky_features = honest_features + ['avg_position_month']
X_leak = feature_frame[leaky_features].fillna(0)
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_leak, y, test_size=0.25, random_state=42, stratify=y)
leaky_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr2, y_tr2)
leaky_auc = roc_auc_score(y_te2, leaky_model.predict_proba(X_te2)[:, 1])
print(f"'Leaky' ROC AUC (label-derived column included): {leaky_auc:.3f}  <- jumps toward perfect")

# Delete the leak, keep the honest number
final_features = honest_features
print(f"\nKeeping honest feature set: {final_features}")
print(f"Honest ROC AUC to report: {honest_auc:.3f}")

Honest ROC AUC (no leak): 0.833
'Leaky' ROC AUC (label-derived column included): 1.000  <- jumps toward perfect

Keeping honest feature set: ['impressions_month', 'clicks_month', 'content_age_days', 'rare_share']
Honest ROC AUC to report: 0.833


Leakage lesson: adding avg_position_month as a feature -- after using it to DEFINE
the label -- pushed the score sharply toward perfect, because the model was just
detecting its own label definition, not finding independent signal. The honest
feature set drops that column, and the honest ROC AUC above is the number I'd
actually report.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

1. Unbalanced history -- clients started tracking at different times; a page with
   "no traffic" early on may just mean tracking hadn't started (check
   dim_clients.gsc_data_start / ga4_data_start before trusting any window).

2. GSC-only early rows -- before a client's ga4_data_start, ga4_data_available is
   FALSE and engagement fields are absent, not zero.

3. Window overlaps -- a fixed calendar month doesn't mean equal history behind
   every content item; trailing history depth varies per client.

4. Correlation only -- this data can show association with future decline, never
   that a refresh WOULD cause recovery, without a real experiment.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.